[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/two_spins/two_spins_space.ipynb)

# Two Spins in the Algebra of Space

This notebook computes exchange entanglement, Bell violations, measurement steering, Zeeman level crossings, and singlet-triplet precession for two coupled spins modelled in two copies of three-dimensional Euclidean space. Starting with one spin up and one spin down, the exchange interaction swaps the two spins while entangling them fully halfway through, where neither spin displays a Bloch vector of its own yet outcomes along matching spatial axes always disagree. A uniform field lowers the aligned triplet below the singlet, a differential field drives oscillations between the singlet and zero-spin triplet, and a closing reflection test reveals how axial Bloch vectors reverse sign in a mirror.

This conventional multiparticle algebra follows Doran and Lasenby's *Geometric Algebra for Physicists* (Cambridge University Press, 2003, Chapter 9) and Havel and Doran's [*Geometric Algebra in Quantum Information Processing*](https://arxiv.org/abs/quant-ph/0004031), tying two spatial copies together in the ideal of `correlator = 0.5 * (1 - xy * XY)` with shared phase generator `reference_spin = correlator * xy`. Reading each spin's turning plane as a spatial vector via `first_pseudoscalar.inverse() * spin` introduces a handedness through `xyz` and `XYZ`, turning the joint spin map into `FirstVector <- SecondVector` while causing mirrored experiments to match the reversed external field. The primary house-style formulation in [two_spins.ipynb](two_spins.ipynb) keeps spins, measurements, and magnetic fields as turning planes in two copies of spacetime so every equation transforms equivariantly under spatial reflection without pseudoscalars.

In bra-ket and Pauli-matrix notation, the pair state is written as a four-component ket $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$ acted upon by Kronecker products $\sigma_i \otimes \sigma_j$ with an explicit imaginary unit $i$, single-spin Bloch vectors require partial traces $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$ followed by $\mathbf a = \operatorname{tr}(\rho_1 \boldsymbol\sigma)$, and joint spin expectations form a separate $3 \times 3$ matrix $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_animation
from examples.quantum.two_spins import render

np.set_printoptions(precision=4, suppress=True)

# Direct-sum algebra of two copies of Euclidean 3D space.
first_space = Algebra("x+y+z+")
second_space = Algebra("X+Y+Z+")
ga = first_space * second_space
context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
FirstVector = ga.gatype.vector().derive.within(first_space)                     # Vector+
SecondVector = ga.gatype.vector().derive.within(second_space)                   # Vector+
FirstPlanes = ga.gatype.bivector().derive.within(first_space)                   # Bivector+
SecondPlanes = ga.gatype.bivector().derive.within(second_space)                 # Bivector+
FirstSpinor = ga.gatype.even().derive.within(first_space)                       # Even+
SecondSpinor = ga.gatype.even().derive.within(second_space)                     # Even+
Spinor = ga.gatype(FirstSpinor * SecondSpinor)                                  # Even+
JointMap = ga.gatype((FirstVector, SecondVector))                               # FirstVector+ <- SecondVector+

# Spatial pseudoscalars of each copy, correlator idempotent, and exchange projectors.
first_pseudoscalar = mv.xyz                                                     # [] Trivector+
second_pseudoscalar = mv.XYZ                                                    # [] Trivector+
correlator = 0.5 * (1 - mv.xy * mv.XY)                                          # [] Spinor+
reference_spin = correlator * mv.xy                                             # [] Spinor+
coupling = mv.yz * mv.YZ + mv.zx * mv.ZX + mv.xy * mv.XY                        # [] Spinor+
singlet_part = 0.25 * (1 + coupling)                                            # [] Spinor+
triplet_part = 0.25 * (3 - coupling)                                            # [] Spinor+

## Two spins in one spatial algebra

Each single-spin state is an even multivector of its own copy of space: spin up along `z` is 1, and a half turn `-ZX` flips the second spin along `-Z` so `-ZX * correlator` prepares the first spin up and the second spin down. Within the ideal of `correlator`, right multiplication by `xy` or `XY` coincides with `reference_spin`, which squares to `-correlator` and generates shared phase rotations from the right while spatial planes of different copies commute. Every normalized pair state has unit scalar part in `2 * state.symmetric_reverse_product()`.

In [ ]:
# First spin up along +z and second spin down along -Z.
up_down = -mv.ZX * correlator                                                   # [] Spinor+
state_norm = 2 * up_down.scalar_norm_squared()                                  # [] Scalar+
print("norm of initial up-down state:", state_norm.to_array())

## Bloch vectors and joint expectations

Projecting the pair spin `2 * (state >> reference_spin)` onto each copy's planes and dividing by `first_pseudoscalar` and `second_pseudoscalar` converts each spin's turning plane into an axial Bloch vector of unit length for product states and shorter length for entangled pairs. Converting both sides of the grade-(2, 2) density block through the two copy pseudoscalars yields `joint_map(state)`, a linear map from the second spin's directions to the first's such that `first_dir | joint(second_dir)` equals the expected product of spin outcomes along `first_dir` and `second_dir`. From its two largest singular values follows the largest Bell combination over all measurement directions.

In bra-ket notation, the Bloch vector of the first spin reads as $\operatorname{tr}(\rho_1 \boldsymbol\sigma)$ for $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$, the joint spin map reads as the spin-pair expectation matrix $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$, and the largest Bell combination reads as $2\sqrt{t_1^2 + t_2^2}$ over the singular values $t_i$ of $T$.

In [ ]:
def bloch_vectors(state: Spinor) -> tuple[FirstVector, SecondVector]:
    """Pseudoscalar division maps each spin's turning-plane projection of `2 * (state >> reference_spin)` into a spatial Bloch vector (`[...] FirstVector+, SecondVector+`)."""
    pair_spin = 2 * (state >> reference_spin)                                   # [...] Bivector+
    first_vector = first_pseudoscalar.inverse() * pair_spin.cast(FirstPlanes)   # [...] FirstVector+
    second_vector = second_pseudoscalar.inverse() * pair_spin.cast(SecondPlanes) # [...] SecondVector+
    return first_vector, second_vector


def joint_map(state: Spinor) -> JointMap:
    """The grade-(2, 2) density block converted through both copy pseudoscalars defines the joint spin map (`[...] FirstVector+ <- SecondVector+`)."""
    density = 2 * state.symmetric_reverse_product()                             # [...] Spinor+
    plane_block = (density * (second_pseudoscalar * SecondVector)).cast(FirstPlanes)   # [...] FirstPlanes+ <- SecondVector+
    return (first_pseudoscalar.inverse() * plane_block).cast(JointMap)          # [...] FirstVector+ <- SecondVector+


start_first, start_second = bloch_vectors(up_down)                              # [] FirstVector+, SecondVector+
start_joint = joint_map(up_down)                                                # [] FirstVector+ <- SecondVector+
bloch_reading = render.Reading(
    stack((mv.x, mv.y, mv.z)),
    stack((mv.X, mv.Y, mv.Z)),
    "each Bloch vector's length",
)
save_figure(render.draw_state(start_first, start_second, bloch_reading), "two_spins_space_state");

On the two Bloch spheres, the red arrow shows the first spin pointing along `+z` with unit length and the blue arrow shows the second spin pointing along `-Z` with unit length.

In [ ]:
def bell_maximum(joint: JointMap) -> Scalar:
    """The two largest singular values of `joint` set the largest Bell combination (`[...] Scalar+`) over all measurement directions."""
    values = joint.svdvals()                                                    # [..., 3] Scalar+
    return 2 * (values[..., 0] ** 2 + values[..., 1] ** 2).square_root()        # [...] Scalar+


start_bell = bell_maximum(start_joint)                                          # [] Scalar+
print("largest Bell combination on initial up-down state:", start_bell.to_array())

## Exchange entanglement

Because `coupling * coupling == 3 + 2 * coupling`, the orthogonal projectors `singlet_part = (1 + coupling) / 4` and `triplet_part = (3 - coupling) / 4` split every state into a singlet eigenstate with eigenvalue 3 and a triplet component with eigenvalue -1. Over exchange angle `angle`, the singlet component turns forward by `3 * angle` in `xy` while the triplet component turns backward by `angle`, swapping `up_down` into `down_up` through a maximally entangled midpoint.

In bra-ket notation, exchange evolution reads as $e^{-i\theta\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2}$, splitting into the singlet projector $\tfrac14\left(I - \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ and the triplet projector $\tfrac14\left(3\,I + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$.

In [ ]:
def exchange_state(state: Spinor, angle: np.ndarray) -> Spinor:
    """Exchange evolution over `angle` turns the singlet part forward by `3 * angle` and the triplet part backward by `angle` (`[...] Spinor+`)."""
    singlet_turn = (mv.xy * (3 * angle)).exp()                                  # [...] Rotor+
    triplet_turn = (mv.xy * -angle).exp()                                       # [...] Rotor+
    return singlet_part * state * singlet_turn + triplet_part * state * triplet_turn   # [...] Spinor+


n_angles = 181
midpoint = n_angles // 2
angles = np.linspace(0.0, np.pi / 4, n_angles)
states = exchange_state(up_down, angles)                                        # [n_angles] Spinor+
first_vectors, second_vectors = bloch_vectors(states)                           # [n_angles] FirstVector+, SecondVector+
joints = joint_map(states)                                                      # [n_angles] FirstVector+ <- SecondVector+
bells = bell_maximum(joints)                                                    # [n_angles] Scalar+
save_figure(render.draw_exchange(angles, bells, first_vectors, bloch_reading), "two_spins_space_exchange");

Across the swap from `0` to `pi / 4`, the red curve on the left shows each Bloch vector shrinking from unit length to zero at the midpoint before returning to unit length, while the purple curve on the right rises past the dashed local bound of 2 to reach the dotted `2 * sqrt(2)` ceiling.

## Singlet state and the Bell bound

Normalizing `singlet_part * up_down` produces the singlet state, whose joint spin map sends every unit direction of the second spin to the opposite direction of the first. Converting unit directions `first_dir` and `second_dir` into planes via `first_pseudoscalar * first_dir` and `second_pseudoscalar * second_dir` builds the left-acting `bell_element`, whose square equals `4 - 4 * (first_choice ^ first_alt) * (second_choice ^ second_alt)`. Because neither wedge of unit vectors exceeds unit norm, the square never exceeds 8 and no quantum state exceeds `2 * sqrt(2)`, which the singlet reaches for orthogonal directions on the first spin paired with diagonal directions on the second.

In bra-ket notation, the Bell element reads as the CHSH operator $\mathcal{B} = A \otimes (B + B') + A' \otimes (B - B')$ with operator square $\mathcal{B}^2 = 4\,I - [A, A'] \otimes [B, B']$, bounding $\langle\mathcal{B}\rangle$ by $2\sqrt{2}$.

In [ ]:
def bell_element(
    first_dir: FirstVector,
    first_alt: FirstVector,
    second_dir: SecondVector,
    second_alt: SecondVector,
) -> Spinor:
    """Two unit directions on each spin combine through the copy pseudoscalars into the left-acting Bell multivector (`[...] Spinor+`)."""
    first_plane = first_pseudoscalar * first_dir                                # [...] FirstPlanes+
    first_alt_plane = first_pseudoscalar * first_alt                            # [...] FirstPlanes+
    return -(
        first_plane * (second_pseudoscalar * (second_dir + second_alt))
        + first_alt_plane * (second_pseudoscalar * (second_dir - second_alt))
    )                                                                           # [...] Spinor+


def expectation(element: Spinor, state: Spinor) -> Scalar:
    """Sandwiching a left-acting multivector `element` from the reverse side gives its scalar expectation (`[...] Scalar+`) in `state`."""
    return 2 * (state << element).select[0]                                     # [...] Scalar+


singlet = np.sqrt(2) * singlet_part * up_down                                   # [] Spinor+
singlet_joint = joint_map(singlet)                                              # [] FirstVector+ <- SecondVector+
rng = np.random.default_rng(0)
first_choice, first_alt = mv(FirstVector, rng.normal(size=(2, 3))).normalized()   # [] FirstVector+, FirstVector+
second_choice, second_alt = mv(SecondVector, rng.normal(size=(2, 3))).normalized() # [] SecondVector+, SecondVector+
sample_element = bell_element(first_choice, first_alt, second_choice, second_alt) # [] Spinor+
element_square = sample_element * sample_element                                # [] Spinor+
wedge_bound = 4 - 4 * (first_choice ^ first_alt) * (second_choice ^ second_alt) # [] Spinor+
half = 1 / np.sqrt(2)
singlet_bell = expectation(bell_element(mv.x, mv.y, -half * (mv.X + mv.Y), -half * (mv.X - mv.Y)), singlet)   # [] Scalar+
print("Bell combination on the singlet:", singlet_bell.to_array(), "  Tsirelson bound:", 2 * np.sqrt(2))

## Steering by measurement

Measuring outcome +1 for the first spin along a unit direction `direction` projects the state onto `0.5 * (state - (first_pseudoscalar * direction) * state * reference_spin)` with probability `0.5 * (1 + (direction | first_vector))`. Renormalizing leaves the second spin with Bloch vector `(second_vector + joint.adjoint()(direction)) / (1 + (direction | first_vector))`, steered by the adjoint of the joint spin map.

In bra-ket notation, the projective measurement applies $\tfrac12(I + \mathbf n\cdot\boldsymbol\sigma)\otimes I$, mapping the second spin's Bloch vector to $(\mathbf b + T^{\mathsf T}\mathbf n)/(1 + \mathbf n\cdot\mathbf a)$.

In [ ]:
def steered_spin(state: Spinor, direction: FirstVector) -> tuple[Scalar, SecondVector]:
    """Projecting the first spin onto outcome +1 along unit `direction` gives the outcome probability (`[...] Scalar+`) and the steered second Bloch vector (`[...] SecondVector+`)."""
    projected_state = 0.5 * (state - (first_pseudoscalar * direction) * state * reference_spin)   # [...] Spinor+
    probability = 2 * projected_state.scalar_norm_squared()                     # [...] Scalar+
    _, second_vector = bloch_vectors(projected_state / probability.square_root())   # [...] SecondVector+
    return probability, second_vector


n_around, n_down = 24, 12
frame_duration_ms = 40
polar = (np.arange(n_down) + 0.5) / n_down * np.pi
azimuth = np.arange(n_around) / n_around * 2 * np.pi
polar_rotors = (mv.zx * (-polar / 2)).exp()                                     # [n_down] Rotor+
azimuth_rotors = (mv.xy * (-azimuth / 2)).exp()                                 # [n_around] Rotor+
polar_vectors = polar_rotors >> mv.z                                            # [n_down] FirstVector+
directions = azimuth_rotors >> polar_vectors[:, None]                           # [n_down, n_around] FirstVector+
probability, steered_vectors = steered_spin(states[:, None, None], directions)  # [n_angles, n_down, n_around] Scalar+, SecondVector+
adjoint_vectors = (second_vectors[:, None, None] + joints[:, None, None].adjoint()(directions)) / (
    1 + (directions | first_vectors[:, None, None])
)                                                                               # [n_angles, n_down, n_around] SecondVector+
show_animation(render.animate_steering(directions, probability, steered_vectors, bloch_reading), "two_spins_space_steering", frame_duration_ms)

RGB-coloured dots on the left sphere mark the sampled measurement directions of the first spin, while matching dots on the right sphere show the steered second Bloch vectors sized by outcome probability, spreading from `-Z` across the whole sphere at maximal entanglement before concentrating at `+Z`.

## Entanglement in the singular values

Local rotors `first_turn` and `second_turn` act on both sides of the joint spin map as `joint_map(first_turn * second_turn * state) == first_turn >> joints(second_turn << SecondVector)`, leaving its singular values invariant: 1 together with the concurrence `concurrence` repeated twice. Each Bloch vector has length `(1 - concurrence**2).square_root()`, while the largest Bell combination equals `2 * (1 + concurrence**2).square_root()`.

In bra-ket notation, local unitaries $U_1 \otimes U_2$ rotate the expectation matrix as $T \mapsto R_1 T R_2^{\mathsf T}$, and pure-state concurrence reads as $C = |\langle\psi|\,\sigma_y\otimes\sigma_y\,|\psi^*\rangle|$.

In [ ]:
first_turn = mv(FirstPlanes, rng.normal(size=3)).exp()                          # [] Rotor+
second_turn = mv(SecondPlanes, rng.normal(size=3)).exp()                        # [] Rotor+
turned_joints = joint_map(first_turn * second_turn * states)                    # [n_angles] FirstVector+ <- SecondVector+
sandwich_joints = first_turn >> joints(second_turn << SecondVector)             # [n_angles] FirstVector+ <- SecondVector+
singular_values = joints.svdvals()                                              # [n_angles, 3] Scalar+
turned_singular_values = turned_joints.svdvals()                                # [n_angles, 3] Scalar+
concurrence = singular_values[..., 1]                                           # [n_angles] Scalar+
save_figure(render.draw_entanglement(angles, singular_values, first_vectors, bloch_reading), "two_spins_space_entanglement");

The solid purple curve stays at 1 for the leading singular value, the dashed purple curve traces the twofold-degenerate concurrence peaking at 1 halfway through the swap, and the solid red curve shows the Bloch vector length vanishing at maximal entanglement.

## Energy levels in a field

In the vector convention, a magnetic field along `z` lowers each spin's energy in proportion to `(first_vector | z) + (second_vector | Z)`. Combined with shifted exchange, the singlet and the three definite-spin triplets have linear field energies whose ground state crosses from the entangled singlet to the spin-aligned triplet at `field == 2 * exchange_rate`.

In bra-ket notation, these levels are the eigenvalues of $H = \tfrac{J}{2}\left(I + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) - \tfrac{B}{2}\left(\sigma_{1z} + \sigma_{2z}\right)$.

In [ ]:
def energy(state: Spinor, exchange_rate: np.ndarray, field: np.ndarray) -> Scalar:
    """Shifted exchange coupling and a uniform `z`-axis field give the expected state energy (`[...] Scalar+`)."""
    first_vector, second_vector = bloch_vectors(state)                          # [...] FirstVector+, SecondVector+
    return exchange_rate * expectation(0.5 * (1 - coupling), state) - 0.5 * field * ((first_vector | mv.z) + (second_vector | mv.Z))   # [...] Scalar+


exchange_rate = np.asarray(1.0)
n_fields = 201
fields = np.linspace(0.0, 4.0, n_fields)
basis_states = stack([
    singlet,
    np.sqrt(2) * triplet_part * up_down,
    correlator,
    mv.zx * mv.ZX * correlator,
])                                                                              # [4] Spinor+
basis_first, basis_second = bloch_vectors(basis_states)                         # [4] FirstVector+, SecondVector+
total_spins = (basis_first | mv.z) + (basis_second | mv.Z)                      # [4] Scalar+
energies = energy(basis_states, exchange_rate, fields[:, None])                 # [n_fields, 4] Scalar+
save_figure(render.draw_levels(fields, energies, total_spins), "two_spins_space_levels");

On the left, the red aligned-triplet energy falls below the purple singlet energy at `field == 2` along the grey ground-state band, causing the ground-state spin on the right to step from 0 to 2.

## Singlet-triplet qubit

A field difference along `z` between neighbouring quantum dots couples the singlet to the zero-spin triplet while leaving the polarized triplets untouched, forming the semiconductor singlet-triplet qubit demonstrated by Petta and coworkers (Science 309, 2180, 2005). Because `time_generator` applied twice multiplies states in this subspace by `-(exchange_rate**2 + difference**2)`, time evolution closes into cosine and sine terms on the qubit sphere.

In bra-ket notation, $-iH$ with $H = \tfrac{J}{2}\left(I + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) + \tfrac{\Delta}{2}\left(\sigma_{1z} - \sigma_{2z}\right)$ yields singlet return probability $1 - \tfrac{\Delta^2}{J^2 + \Delta^2}\sin^2\!\left(\sqrt{J^2 + \Delta^2}\,t\right)$.

In [ ]:
def time_generator(state: Spinor, exchange_rate: np.ndarray, difference: np.ndarray) -> Spinor:
    """Shifted exchange about `reference_spin` and differential field `difference` drive the infinitesimal time rate (`[...] Spinor+`) of `state`."""
    return -exchange_rate * (0.5 * (1 - coupling) * state) * reference_spin - difference * 0.5 * (mv.xy - mv.XY) * state   # [...] Spinor+


def qubit_state(state: Spinor, exchange_rate: np.ndarray, difference: np.ndarray, times: np.ndarray) -> Spinor:
    """Harmonic precession under `exchange_rate` and `difference` carries a zero-net-spin `state` to `times` (`[...] Spinor+`)."""
    rate = np.sqrt(exchange_rate**2 + difference**2)
    step_generator = time_generator(state, exchange_rate, difference)           # [...] Spinor+
    return np.cos(rate * times) * state + np.sin(rate * times) / rate * step_generator   # [...] Spinor+


difference = np.asarray(1.0)
exchange_rates = np.array([0.0, 1.0, 3.0])
n_times = 241
times = np.linspace(0.0, np.pi, n_times)
qubit_states = qubit_state(singlet, exchange_rates[:, None], difference, times) # [3, n_times] Spinor+
singlet_return = expectation(singlet_part, qubit_states)                        # [3, n_times] Scalar+
subspace_pair = basis_states[:2]                                                # [2] Spinor+
repeated_generator = time_generator(
    time_generator(subspace_pair, exchange_rates[1], difference),
    exchange_rates[1],
    difference,
)                                                                               # [2] Spinor+

In [ ]:
def qubit_coordinates(state: Spinor) -> tuple[Scalar, Scalar, Scalar]:
    """Spin imbalance, relative phase, and singlet-triplet balance place `state` on the two-spin qubit sphere (`[...] Scalar+` each)."""
    first_vector, second_vector = bloch_vectors(state)                          # [...] FirstVector+, SecondVector+
    imbalance = 0.5 * ((first_vector | mv.z) - (second_vector | mv.Z))          # [...] Scalar+
    phase_coordinate = expectation(-0.25 * (1 - coupling) * (mv.xy - mv.XY), state)   # [...] Scalar+
    singlet_balance = expectation(0.5 * (coupling - 1), state)                  # [...] Scalar+
    return imbalance, phase_coordinate, singlet_balance


imbalance, phase_coordinate, singlet_balance = qubit_coordinates(qubit_states)  # [3, n_times] Scalar+, Scalar+, Scalar+
qubit_first, qubit_second = bloch_vectors(qubit_states)                         # [3, n_times] FirstVector+, SecondVector+
save_figure(render.draw_qubit(imbalance, phase_coordinate, singlet_balance, times, singlet_return, exchange_rates), "two_spins_space_qubit");

From the black dot at the singlet pole, purple traces pure differential driving through the zero-spin triplet, red traces equal exchange and differential driving through `up-down`, and blue confines precession near the singlet pole, matching the return-probability curves on the right.

## Behavior in a mirror

A spatial mirror reflects every turning plane into its mirror image, acting on the pair state from the left as the half turn in the plane orthogonal to the mirror normal. Reading turning planes as polar vectors via `first_pseudoscalar = xyz` breaks that reflection symmetry because a mirror negates `xyz`, making the Bloch vector of the mirrored state equal minus the reflected Bloch vector. Consequently, projective measurements along reflected directions yield complementary probabilities summing to 1 instead of matching probabilities, and a mirror whose normal is `x` or `y` leaves a `z`-vector magnetic field unchanged while flipping the `xy` turning plane, giving the mirrored pair the energies and qubit precession of the reversed field. Formulating spins, fields, and measurements directly as turning planes in [two_spins.ipynb](two_spins.ipynb) avoids spatial pseudoscalars and restores exact mirror equivariance.

In [ ]:
# Mirrors with normals x and y across the z-axis field, acting on both copies simultaneously.
reflections = stack([mv.x * mv.X, mv.y * mv.Y])                                 # [2] Versor+
half_turns = stack([mv.yz * mv.YZ, mv.zx * mv.ZX])                              # [2] Spinor+
mirrored_states = half_turns[:, None] * states                                  # [2, n_angles] Spinor+
mirrored_first, mirrored_second = bloch_vectors(mirrored_states)                # [2, n_angles] FirstVector+, SecondVector+
reflected_first = reflections[:, None] >> first_vectors                         # [2, n_angles] FirstVector+
reflected_second = reflections[:, None] >> second_vectors                       # [2, n_angles] SecondVector+
mirrored_probability, _ = steered_spin(
    mirrored_states[:, :, None, None],
    reflections[:, None, None, None] >> directions,
)                                                                               # [2, n_angles, n_down, n_around] Scalar+
mirrored_energies = energy(half_turns[:, None] * basis_states, exchange_rate, fields[:, None, None])   # [n_fields, 2, 4] Scalar+
reversed_energies = energy(basis_states, exchange_rate, -fields[:, None])       # [n_fields, 4] Scalar+
mirrored_qubit_states = qubit_state(half_turns[:, None, None] * singlet, exchange_rates[:, None], difference, times)   # [2, 3, n_times] Spinor+
reversed_qubit_states = half_turns[:, None, None] * qubit_state(singlet, exchange_rates[:, None], -difference, times)  # [2, 3, n_times] Spinor+

initial_alignment = (mirrored_first[0, 0] | reflected_first[0, 0])              # [] Scalar+
print("dot product of mirrored Bloch vector with reflected Bloch vector:", initial_alignment.to_array())
print("mirrored energies at field == 2:", mirrored_energies[n_fields // 2, 0].to_array(), "  unmirrored:", energies[n_fields // 2].to_array())

In [ ]:
# checks
# Correlator, reference spin, exchange projectors, swap endpoints, and Bell bound.
assert ((correlator * correlator - correlator).scalar_norm_squared() < 1e-12).all()
assert ((correlator * mv.xy - correlator * mv.XY).scalar_norm_squared() < 1e-12).all()
assert ((reference_spin * reference_spin + correlator).scalar_norm_squared() < 1e-12).all()
assert ((state_norm - 1).abs() < 1e-12).all()
assert ((coupling * coupling - 3 - 2 * coupling).scalar_norm_squared() < 1e-12).all()
assert ((singlet_part * singlet_part - singlet_part).scalar_norm_squared() < 1e-12).all()
assert ((triplet_part * triplet_part - triplet_part).scalar_norm_squared() < 1e-12).all()
assert ((singlet_part + triplet_part - 1).scalar_norm_squared() < 1e-12).all()
assert ((2 * states.scalar_norm_squared() - 1).abs() < 1e-7).all()
assert ((start_bell - 2).abs() < 1e-12).all()
assert (first_vectors[midpoint].scalar_norm_squared() < 1e-7).all()
assert ((bells[0] - 2).abs() < 1e-7).all()
assert ((bells[midpoint] - 2 * np.sqrt(2)).abs() < 1e-7).all()
assert ((bells[-1] - 2).abs() < 1e-7).all()
assert ((first_vectors[-1] + mv.z).scalar_norm_squared() < 1e-7).all()
assert ((element_square - wedge_bound).scalar_norm_squared() < 1e-12).all()
assert ((singlet_bell - 2 * np.sqrt(2)).abs() < 1e-12).all()
for second_unit, first_unit in ((mv.X, mv.x), (mv.Y, mv.y), (mv.Z, mv.z)):
    assert ((singlet_joint(second_unit) + first_unit).scalar_norm_squared() < 1e-12).all()

# Steering adjoint formula, singular-value invariance, field crossing, and qubit precession.
assert ((steered_vectors - adjoint_vectors).scalar_norm_squared() < 1e-9).all()
assert ((probability - 0.5 * (1 + (directions | first_vectors[:, None, None]))).abs() < 1e-11).all()
assert ((turned_joints - sandwich_joints).svdvals().abs() < 1e-9).all()
assert ((turned_singular_values - singular_values).abs() < 1e-9).all()
assert ((singular_values[:, 0] - 1).abs() < 1e-9).all()
assert (((first_vectors | first_vectors) + concurrence**2 - 1).abs() < 1e-9).all()
assert ((total_spins - np.array([0.0, 0.0, 2.0, -2.0])).abs() < 1e-12).all()
assert ((energies[n_fields // 2] - np.array([-1.0, 1.0, -1.0, 3.0])).abs() < 1e-12).all()
equal_rate = np.hypot(exchange_rates[1], difference)
assert ((repeated_generator + equal_rate**2 * subspace_pair).scalar_norm_squared() < 1e-12).all()
generalized_rates = np.hypot(exchange_rates[:, None], difference)
expected_return = 1 - (difference / generalized_rates) ** 2 * np.sin(generalized_rates * times) ** 2
assert ((singlet_return - expected_return).abs() < 1e-12).all()
assert (qubit_first[0].scalar_norm_squared() < 1e-12).all()
assert ((imbalance * imbalance + phase_coordinate * phase_coordinate + singlet_balance * singlet_balance - 1).abs() < 1e-12).all()

# Axial sign inversions in the conventional Bloch-vector reading under spatial reflections.
assert (((reflections >> first_pseudoscalar) + first_pseudoscalar).scalar_norm_squared() < 1e-12).all()
assert ((mirrored_first + reflected_first).scalar_norm_squared() < 1e-12).all()
assert ((mirrored_second + reflected_second).scalar_norm_squared() < 1e-12).all()
assert ((mirrored_probability + probability - 1).abs() < 1e-12).all()
assert ((mirrored_energies - reversed_energies[:, None]).abs() < 1e-12).all()
assert ((mirrored_qubit_states - reversed_qubit_states).scalar_norm_squared() < 1e-12).all()